### 1. Construct initial test text

In [20]:
filename = 'textV1.txt'

with open(filename, 'w') as file_object:
    file_object.write("low low low low low\n")
    file_object.write("lower lower widest widest widest\n")
    file_object.write("newest newest newest newest newest newest\n")

### 2. Initialization: GPT2-style character mapping function

In [21]:
def gpt2_bytes_to_unicode_local(): # Use local name to avoid potential conflicts
    """
    Convert bytes to Unicode characters. 
    Calling the function directly returns a dictionary in the format {number: unicode character}.
    """
    # bs: byte values (0-255)
    bs = (
        list(range(ord("!"), ord("~") + 1))
        + list(range(ord("¡"), ord("¬") + 1))
        + list(range(ord("®"), ord("ÿ") + 1))
    )
    # cs: corresponding unicode code points
    cs = bs[:]

    n = 0
    m = 0
    for b in range(2**8):
        if b not in bs:
            bs.append(b)
            cs.append(2**8 + n)
            n += 1
    cs = [chr(n) for n in cs]
    
    return dict(zip(bs, cs))

In [22]:
def validate_special_tokens(
    vocab: dict[int, bytes],
    special_tokens: list[str],
) -> list[str]:
    """
    Verify the validity of special characters and incorporate them into the original mapping dictionary.

    Args:
        vocab (dict[int, str]): Original dictionary
        special_tokens (list[str]): A list of string special tokens to be added to the tokenizer vocabulary.

    Returns:
        list[str],
            vocab_special_validated: Validated initial dictionary with special characters.
    """

    seen = set()
    current_next_id: int = 256 # New token IDs start from 256.
    for token in special_tokens:
        # Check for empty strings
        if not token:
            raise ValueError("Special symbols cannot be empty strings.")
        
        # Check for duplicate special symbols.
        if token in seen:
            raise ValueError(f"Duplicate special symbols.: '{token}'")
        seen.add(token)
        
        # Check if it has already been mapped in the initial vocabulary.
        if len(token) == 1 and ord(token) < 256:
            raise ValueError(f"Duplicate special tokens in the initial vocabulary.: '{token}'")

        # No issue, add it to the initial vocabulary.
        vocab[current_next_id] = token.encode("utf-8")
        current_next_id += 1

    return vocab

Check the generated mapping tuples

In [23]:
# Initialization mapping
special_tokens=["<|endoftext|>"]
vocab: dict[int, bytes] = validate_special_tokens({i: bytes([i]) for i in range(256)}, special_tokens)
_BYTES_TO_UNICODE_MAP = gpt2_bytes_to_unicode_local()
_UNICODE_TO_BYTES_MAP = {v: bytes([k]) for k, v in _BYTES_TO_UNICODE_MAP.items()}


_BYTES_TO_UNICODE_MAP,vocab,_UNICODE_TO_BYTES_MAP

({33: '!',
  34: '"',
  35: '#',
  36: '$',
  37: '%',
  38: '&',
  39: "'",
  40: '(',
  41: ')',
  42: '*',
  43: '+',
  44: ',',
  45: '-',
  46: '.',
  47: '/',
  48: '0',
  49: '1',
  50: '2',
  51: '3',
  52: '4',
  53: '5',
  54: '6',
  55: '7',
  56: '8',
  57: '9',
  58: ':',
  59: ';',
  60: '<',
  61: '=',
  62: '>',
  63: '?',
  64: '@',
  65: 'A',
  66: 'B',
  67: 'C',
  68: 'D',
  69: 'E',
  70: 'F',
  71: 'G',
  72: 'H',
  73: 'I',
  74: 'J',
  75: 'K',
  76: 'L',
  77: 'M',
  78: 'N',
  79: 'O',
  80: 'P',
  81: 'Q',
  82: 'R',
  83: 'S',
  84: 'T',
  85: 'U',
  86: 'V',
  87: 'W',
  88: 'X',
  89: 'Y',
  90: 'Z',
  91: '[',
  92: '\\',
  93: ']',
  94: '^',
  95: '_',
  96: '`',
  97: 'a',
  98: 'b',
  99: 'c',
  100: 'd',
  101: 'e',
  102: 'f',
  103: 'g',
  104: 'h',
  105: 'i',
  106: 'j',
  107: 'k',
  108: 'l',
  109: 'm',
  110: 'n',
  111: 'o',
  112: 'p',
  113: 'q',
  114: 'r',
  115: 's',
  116: 't',
  117: 'u',
  118: 'v',
  119: 'w',
  120: 'x',
  121: 'y'

### 3. Text Preprocessing

In [28]:
import regex
from collections import defaultdict

In [32]:
input_path = filename

try:
    with open(input_path, "r", encoding="utf-8", errors="ignore") as f:
        text = f.read() # 读取整个文件内容
except FileNotFoundError:
    text = "" # 如果文件不存在，视为空文本处理

token_frequency_table = defaultdict(int)
chunks = regex.split('|'.join(map(regex.escape,special_tokens)),text) #首先按照特殊字符进行大分割，比如<endoftext>按照章节分割
# 然后在大分割里小分割，按照空格和标点
PAT = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
for chunk in chunks:
    for word in regex.findall(PAT, chunk):
        word_bytes = word.encode("utf-8") #对每一个单词进行编码，并转换为bytes
        bytes_list = [bytes([x]) for x in word_bytes] #e.g. ['h', 'e', 'l', 'l', 'o']
        token_frequency_table[tuple(bytes_list)] += 1 #统计每个token出现的频率

bytes_list, "", token_frequency_table

([b'\n'],
 '',
 defaultdict(int,
             {(b'l', b'o', b'w'): 1,
              (b' ', b'l', b'o', b'w'): 4,
              (b'\n',): 3,
              (b'l', b'o', b'w', b'e', b'r'): 1,
              (b' ', b'l', b'o', b'w', b'e', b'r'): 1,
              (b' ', b'w', b'i', b'd', b'e', b's', b't'): 3,
              (b'n', b'e', b'w', b'e', b's', b't'): 1,
              (b' ', b'n', b'e', b'w', b'e', b's', b't'): 5}))

In [12]:
# 然后把“单词”转换为初始的unicode字符序列,[[u'h', u'e', u'l', u'l', u'o'], [u'w', u'o', u'r', u'l', u'd']]
unicode_sequences: list[list[str]] = []

for word_str in raw_words:
    word_as_raw_bytes: bytes = word_str.encode("utf-8") # 把每个单词都转为字节串形式
    if not word_as_raw_bytes: # 跳过空字符串（可能由多个连续空格产生）
        continue
    unicode_sequences.append([byte_to_unicode[byte_val] for byte_val in word_as_raw_bytes]) # 将单词映射为unicode字符，并添加到unicode_sequences中

merges: list[tuple[bytes, bytes]] = [] # 用于存储合并操作记录

unicode_sequences

[['l', 'o', 'w'],
 ['Ġ', 'l', 'o', 'w'],
 ['Ġ', 'l', 'o', 'w'],
 ['Ġ', 'l', 'o', 'w'],
 ['Ġ', 'l', 'o', 'w'],
 ['Ċ', 'l', 'o', 'w', 'e', 'r'],
 ['Ġ', 'l', 'o', 'w', 'e', 'r'],
 ['Ġ', 'w', 'i', 'd', 'e', 's', 't'],
 ['Ġ', 'w', 'i', 'd', 'e', 's', 't'],
 ['Ġ', 'w', 'i', 'd', 'e', 's', 't'],
 ['Ċ', 'n', 'e', 'w', 'e', 's', 't'],
 ['Ġ', 'n', 'e', 'w', 'e', 's', 't'],
 ['Ġ', 'n', 'e', 'w', 'e', 's', 't'],
 ['Ġ', 'n', 'e', 'w', 'e', 's', 't'],
 ['Ġ', 'n', 'e', 'w', 'e', 's', 't'],
 ['Ġ', 'n', 'e', 'w', 'e', 's', 't']]

In [13]:
import collections
def get_stats(token_sequences: list[list[str]]) -> collections.Counter:
    """
    统计token序列中所有相邻unicode字符对的频率
    """
    pair_counts = collections.Counter()
    for sequence in token_sequences:
        for i in range(len(sequence) - 1):
            pair = (sequence[i], sequence[i+1])
            pair_counts[pair] += 1
    return pair_counts

In [14]:
def merge_pair_in_sequences(
    token_sequences: list[list[str]],
    pair_to_merge: tuple[str, str],
    new_token_representation: str
) -> list[list[str]]:
    """
    在token序列中用新的unicode字符表示替换指定的字节对
    用 new_token_representation 替换所有出现的 pair_to_merge。
    假设：
    token_sequences = [['h', 'e', 'l', 'l', 'o']]
    pair_to_merge = ('l', 'l')
    new_token_representation = 'll'
    处理过程：
    遍历 ['h', 'e', 'l', 'l', 'o']
    前两个字节不是 ('l', 'l')，跳过
    到了下标2和3，发现是 ('l', 'l')，合并成 'll'
    结果变成 ['h', 'e', 'll', 'o']
    """
    new_overall_sequences = []
    (p1, p2) = pair_to_merge
    for sequence in token_sequences:
        new_sequence = []
        i = 0
        while i < len(sequence):
            if i < len(sequence) - 1 and sequence[i] == p1 and sequence[i+1] == p2:
                new_sequence.append(new_token_representation)
                i += 2
            else:
                new_sequence.append(sequence[i])
                i += 1
        new_overall_sequences.append(new_sequence)
    return new_overall_sequences


In [ ]:
import json

vocab: dict[int, bytes] = {i: bytes([i]) for i in range(256)}

vocab, byte_to_unicode


In [ ]:
merges: list[tuple[bytes, bytes]] = []

vocab_size = 260

# 第四步开始训练BPE算法
while len(vocab) < vocab_size: # 添加新的token直到词汇表达到指定大小
        if not unicode_sequences: # 如果没有数据可以处理了
            break
        pair_counts = get_stats(unicode_sequences) # 统计当前所有unicode字符对的频率
        if not pair_counts: # 如果没有可以合并的unicode字符对了
            break
        
        # 找到频率最高的unicode字符对
        best_pair: tuple[str, str] = max(pair_counts, key=lambda x: pair_counts[x])

        # 将最佳unicode字符对的两个unicode字符连接起来
        new_token_str: str = best_pair[0] + best_pair[1]

        #将unicode字符转换为字节token
        p1_bytes = token_str_to_bytes[best_pair[0]]
        p2_bytes = token_str_to_bytes[best_pair[1]]  
        # 将新token添加到词汇表，并记录这次合并操作
        new_token_bytes: bytes = p1_bytes + p2_bytes
        token_str_to_bytes[new_token_str] = new_token_bytes
        vocab[current_next_id] = new_token_bytes

        merges.append((p1_bytes, p2_bytes))

        # 更新训练数据中的所有序列：用新token替换掉原来的字节对
        unicode_sequences = merge_pair_in_sequences(unicode_sequences, best_pair, new_token_str)
        
        current_next_id += 1 # 为下一个可能的新token准备ID

    # print(merges[:10])
    # 保存词汇表到文件
with open("vocab.json", "w", encoding="utf-8") as f:
    vocab_dict = {token_id: token_bytes.decode("utf-8", errors="replace") 
                for token_id, token_bytes in vocab.items()}
    json.dump(vocab_dict, f, ensure_ascii=False, indent=4)
    
    # 保存合并操作记录到文件
with open("merges.txt", "w", encoding="utf-8") as f:
    for p1, p2 in merges:
        # 将字节对转换为可读的字符串表示
        p1_str = p1.decode("utf-8", errors="replace")
        p2_str = p2.decode("utf-8", errors="replace")
        f.write(f"{p1_str} {p2_str}\n")
vocab, merges # 返回最终的词汇表和合并记录